# 01 — Fondation : explorer le corpus

On découvre le corpus **IEMP Malian Jembe** (Polak) : 3 pièces (Manjanin, Maraka, Woloso), pistes **isolées** par instrument + onsets + **grille métrique**.

Tout passe par le module partagé `djembe_utils` (`import djembe_utils as dj`). Ce notebook pose les bases pour les 4 objectifs : **frappes, rôles, nombre, rythme**.

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import djembe_utils as dj

ROLE_COLOR = {'jembe_solo':'#d63bd6', 'jembe_accomp':'#9b59b6', 'dunun':'#2b6cff'}
CODE_COLOR = {'J1':'#d63bd6', 'J2':'#9b59b6', 'D1':'#2b6cff', 'D2':'#1a3a8a'}
print('Pièces :', dj.PIECES)

## 1. Vue d'ensemble
Combien de pistes, quels rôles, quel effectif par prise.

In [ ]:
c = dj.list_corpus()
print('Pistes isolées :', len(c), '\n')
print('Composition (pièce × rôle) :')
display(c.groupby(['piece','role']).size().unstack(fill_value=0))
print('Effectif par prise (nb instruments) :')
display(c.groupby(['piece','take']).size().rename('n').reset_index()
          .pivot(index='take', columns='piece', values='n').fillna('-'))

## 2. Explorer une prise : qui joue quand
On superpose les onsets de chaque instrument (couleur = rôle) et on écoute. Change `PIECE` / `TAKE`.

In [ ]:
PIECE, TAKE = 'Maraka', 1
codes = c[(c['piece']==PIECE) & (c['take']==TAKE)]['code'].tolist()

plt.figure(figsize=(14, 0.7*len(codes)+1))
for i, code in enumerate(codes):
    on = dj.load_onsets(PIECE, TAKE, code)
    ot = on[on['Time'] < 8]['Time']
    plt.vlines(ot, i, i+0.8, color=CODE_COLOR[code])
plt.yticks([i+0.4 for i in range(len(codes))], [f'{code} · {dj.ROLE[code]}' for code in codes])
plt.xlabel('temps (s)'); plt.title(f'{PIECE}_{TAKE} — onsets par instrument (8 s)'); plt.show()

# écoute du mélange (somme des pistes isolées)
tracks = [dj.load_track(PIECE, TAKE, code) for code in codes]
n = min(map(len, tracks)); mix = sum(t[:n] for t in tracks)
print('Mélange (toutes pistes) :'); display(Audio(mix, rate=dj.SR))

## 3. La grille métrique = la signature du rythme

Chaque cycle a **12 subdivisions**. Pour chaque instrument, on regarde **à quelles subdivisions il frappe** (probabilité sur tous les cycles). C'est le motif d'accompagnement — la base pour reconnaître un rythme.

In [ ]:
M = np.vstack([dj.subdivision_profile(PIECE, TAKE, code).values for code in codes])
plt.figure(figsize=(11, 0.6*len(codes)+1))
plt.imshow(M, aspect='auto', cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='proba de frappe')
plt.xticks(range(12), range(1,13)); plt.yticks(range(len(codes)), [f'{c0} · {dj.ROLE[c0]}' for c0 in codes])
plt.xlabel('subdivision du cycle (1–12)'); plt.title(f'{PIECE}_{TAKE} — motif métrique par instrument'); plt.show()

## 4. Qu'est-ce qui distingue un rythme ?

On moyenne le motif de chaque instrument sur les prises de chaque pièce. Découverte importante :
- le **jembé d'accompagnement (J2)** joue le **même squelette ternaire** dans les 3 pièces (il ne distingue pas le rythme) ;
- le **soliste (J1)** joue un peu partout (trop varié pour signer) ;
- c'est le **dunun (D1)** qui porte la **signature** du rythme.

→ pour reconnaître un rythme (notebook `05_rythme`), on s'appuiera donc sur le **motif du dunun**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for code, a, title in [('D1', ax[0], 'DUNUN (D1) — distingue les rythmes'),
                       ('J2', ax[1], 'JEMBE accomp. (J2) — squelette partage')]:
    for piece in dj.PIECES:
        takes = c.loc[(c['piece']==piece) & (c['code']==code), 'take'].unique()
        profs = [dj.subdivision_profile(piece, t, code) for t in takes]
        profs = [p.values for p in profs if p is not None]
        if not profs:
            continue
        a.plot(range(1, 13), np.mean(profs, axis=0), marker='o', lw=2, label=piece)
    a.set_xticks(range(1, 13)); a.set_title(title); a.set_xlabel('subdivision (1-12)')
    a.grid(alpha=.2); a.legend()
ax[0].set_ylabel('proba de frappe')
plt.tight_layout(); plt.show()

## Bilan & suite

- Corpus chargé proprement via `djembe_utils` (46 pistes, 3 pièces, rôles connus).
- **Découverte** : le jembé d'accompagnement (J2) joue un squelette ternaire **commun** aux 3 pièces ; c'est le **dunun (D1)** qui porte l'identité du rythme, et le soliste (J1) est trop varié pour signer. → l'étape *rythme* s'appuiera sur le motif du dunun.
- **Prochain notebook `02_frappes`** : sur les jembés isolés, séparer **bass / tone / slap**.